# 📓 02: Model Training and Evaluation Notebook

This notebook covers testing various machine learning configurations, tuning TF-IDF parameters, and comparing classification results for our sentiment models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

sys.path.append(os.path.abspath("../src"))
from preprocessor import Preprocessor

## 1. Load and Preprocess Data

In [ ]:
df = pd.read_csv("../data/raw_tweets.csv").dropna(subset=["text"])
preprocessor = Preprocessor()

print("Applying preprocessor cleaning...")
df['clean_text'] = df['text'].apply(lambda x: preprocessor.preprocess_as_string(x))
df = df[df['clean_text'].str.strip() != ""]

df["label"] = df["sentiment"].replace({0: "Negative", 4: "Positive", "0": "Negative", "4": "Positive"})
df.head()

## 2. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.20, random_state=42, stratify=df['label']
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## 3. Fit Vectorizer and Classifiers

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Fit Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

# Fit Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_vec, y_train)

## 4. Evaluation and Matrix Plotting

In [ ]:
for name, model in [("Naive Bayes", nb_model), ("Logistic Regression", lr_model)]:
    preds = model.predict(X_test_vec)
    print(f"=== {name} Classification Report ===")
    print(classification_report(y_test, preds))
    
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=model.classes_, yticklabels=model.classes_)
    plt.title(f'{name} Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()